# Ingestion Smoke Test

This notebook verifies that the Neuromatch Allen Visual Behavior preprocessed parquet can be downloaded, loaded, summarized, and plotted.

## Colab setup

Run the cells top-to-bottom. The setup cell below installs missing Python dependencies inside the notebook, uses the repo package when `src/` is available, and falls back to notebook-local helpers if this notebook is opened standalone in Colab.

In [ ]:
import importlib.util
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path

try:
    import google.colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "nma_data_ingestion").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

if IN_COLAB:
    package_to_module = {
        "pandas": "pandas",
        "pyarrow": "pyarrow",
        "requests": "requests",
        "tqdm": "tqdm",
        "matplotlib": "matplotlib",
    }
    missing = [pkg for pkg, mod in package_to_module.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import pandas as pd
import requests
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    from nma_data_ingestion import (
        download_neuromatch_dataset,
        load_neuromatch_dataset,
        validate_neuromatch_dataset,
    )
    USING_REPO_PACKAGE = True
except ModuleNotFoundError:
    USING_REPO_PACKAGE = False
    NEUROMATCH_URL = "https://ndownloader.figshare.com/files/28470255"
    NEUROMATCH_FILENAME = "allen_visual_behavior_2p_change_detection_familiar_novel_image_sets.parquet"
    REQUIRED_COLUMNS = (
        "trace",
        "trace_timestamps",
        "image_name",
        "cell_specimen_id",
        "rewarded",
        "omitted",
        "is_change",
        "cre_line",
        "ophys_session_id",
        "ophys_experiment_id",
    )

    @dataclass(frozen=True)
    class DatasetSummary:
        path: Path | None
        rows: int
        columns: int
        sessions: int
        experiments: int
        cre_lines: tuple[str, ...]
        omitted_counts: dict[str, int]
        is_change_counts: dict[str, int]
        rewarded_counts: dict[str, int]

    def download_neuromatch_dataset(data_dir: str | Path = "data/raw", *, validate: bool = False, overwrite: bool = False) -> Path:
        data_dir = Path(data_dir)
        data_dir.mkdir(parents=True, exist_ok=True)
        target_path = data_dir / NEUROMATCH_FILENAME
        partial_path = target_path.with_suffix(target_path.suffix + ".part")
        if target_path.exists() and not overwrite:
            if validate:
                validate_neuromatch_dataset(target_path)
            return target_path
        if partial_path.exists():
            partial_path.unlink()
        response = requests.get(NEUROMATCH_URL, stream=True, timeout=60, allow_redirects=True)
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with partial_path.open("wb") as file_obj:
            with tqdm(total=total or None, unit="B", unit_scale=True, desc=NEUROMATCH_FILENAME) as progress:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        file_obj.write(chunk)
                        progress.update(len(chunk))
        partial_path.replace(target_path)
        if validate:
            validate_neuromatch_dataset(target_path)
        return target_path

    def load_neuromatch_dataset(path: str | Path) -> pd.DataFrame:
        path = Path(path)
        if not path.exists():
            raise FileNotFoundError(f"Dataset not found at {path}.")
        return pd.read_parquet(path)

    def _value_counts(values) -> dict[str, int]:
        counts = pd.Series(values).value_counts(dropna=False).sort_index()
        return {str(index): int(count) for index, count in counts.items()}

    def validate_neuromatch_dataset(data_or_path) -> DatasetSummary:
        if isinstance(data_or_path, pd.DataFrame):
            path = None
            data = data_or_path
        else:
            path = Path(data_or_path)
            data = load_neuromatch_dataset(path)
        missing = sorted(set(REQUIRED_COLUMNS) - set(data.columns))
        if missing:
            raise ValueError(f"Dataset is missing required column(s): {', '.join(missing)}")
        if data.empty:
            raise ValueError("Dataset is empty.")
        return DatasetSummary(
            path=path,
            rows=len(data),
            columns=len(data.columns),
            sessions=int(data["ophys_session_id"].nunique(dropna=True)),
            experiments=int(data["ophys_experiment_id"].nunique(dropna=True)),
            cre_lines=tuple(sorted(str(value) for value in data["cre_line"].dropna().unique())),
            omitted_counts=_value_counts(data["omitted"]),
            is_change_counts=_value_counts(data["is_change"]),
            rewarded_counts=_value_counts(data["rewarded"]),
        )

DATA_DIR = PROJECT_ROOT / "data" / "raw"
print(f"Using repo package: {USING_REPO_PACKAGE}")
print(f"Data directory: {DATA_DIR}")

In [ ]:
dataset_path = download_neuromatch_dataset(data_dir=DATA_DIR, validate=True)
dataset_path

In [ ]:
data = load_neuromatch_dataset(dataset_path)
summary = validate_neuromatch_dataset(data)
summary

In [ ]:
print(f"rows: {summary.rows:,}")
print(f"columns: {summary.columns:,}")
print(f"sessions: {summary.sessions:,}")
print(f"experiments: {summary.experiments:,}")
print(f"cre lines: {summary.cre_lines}")
print(f"omitted counts: {summary.omitted_counts}")
print(f"change counts: {summary.is_change_counts}")
print(f"rewarded counts: {summary.rewarded_counts}")

In [ ]:
example_cells = data.drop_duplicates("cell_specimen_id").head(4)["cell_specimen_id"]

fig, ax = plt.subplots(figsize=(8, 4))
for cell_id in example_cells:
    row = data[data["cell_specimen_id"] == cell_id].iloc[0]
    ax.plot(row["trace_timestamps"], row["trace"], label=str(cell_id))

ax.axvline(0, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("Time from stimulus or omission onset (s)")
ax.set_ylabel("dF/F")
ax.legend(title="cell_specimen_id", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()